# Рекомендаційні системи
**Бізнес-контекст:** компанія BookSphere тестує поліпшену систему рекомендацій книжок, яка об'єднує контентну та колаборативну моделі.

**Два підходи:**
- **Контентна модель** рекомендує книжки, схожі за описом (тут — за назвою й авторами): ми шукаємо тексти з близькими TF-IDF-векторами.
- **Колаборативна фільтрація** ігнорує зміст і використовує поведінку користувачів: SVD розкладає матрицю оцінок «користувач × книжка» на приховані фактори і передбачає оцінку, яку користувач поставив би книжці, якої ще не читав.

In [2]:
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

## Завдання 1. Підготовка даних


In [3]:
BASE = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/"
books = pd.read_csv(BASE + "books.csv")
ratings = pd.read_csv(BASE + "ratings.csv")
print("books:", books.shape, " ratings:", ratings.shape)

books: (10000, 23)  ratings: (5976479, 3)


In [4]:
books = books[["book_id", "title", "authors"]].dropna().drop_duplicates()
ratings = ratings[["user_id", "book_id", "rating"]].dropna().drop_duplicates()
# один користувач — одна оцінка однієї книжки
ratings = ratings.drop_duplicates(subset=["user_id", "book_id"])
books = books.drop_duplicates(subset="book_id")

df = ratings.merge(books, on="book_id", how="inner")[["book_id", "title", "authors", "user_id", "rating"]]
print("Після очищення та об'єднання:", df.shape)
print("Користувачів:", df["user_id"].nunique(), " Книжок:", df["book_id"].nunique())
df.head()

Після очищення та об'єднання: (5976479, 5)
Користувачів: 53424  Книжок: 10000


,book_id,title,authors,user_id,rating
0,258,The Shadow of the Wind (The Cemetery of Forgot...,"Carlos Ruiz Zafón, Lucia Graves",1,5
1,4081,I am Charlotte Simmons,Tom Wolfe,2,4
2,260,How to Win Friends and Influence People,Dale Carnegie,2,5
3,9296,The Drama of the Gifted Child: The Search for ...,"Alice Miller, Ruth Ward",2,5
4,2318,The Millionaire Next Door: The Surprising Secr...,"Thomas J. Stanley, William D. Danko",2,3


## Завдання 2. Контентна модель


In [5]:
books = books.reset_index(drop=True)
books["text"] = books["title"] + " " + books["authors"]

tfidf = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(books["text"])
print("TF-IDF матриця:", tfidf_matrix.shape)

TF-IDF матриця: (10000, 55171)


In [6]:
def find_book_index(title):
    """Точний збіг (без регістру), інакше перший, що містить фрагмент назви."""
    low = books["title"].str.lower()
    exact = books.index[low == title.lower()]
    if len(exact):
        return exact[0]
    part = books.index[low.str.contains(title.lower(), regex=False)]
    if len(part):
        return part[0]
    raise ValueError(f"Книжку '{title}' не знайдено")

def recommend(title, n=5):
    """n найсхожіших книжок за TF-IDF(title + authors) і косинусною схожістю."""
    idx = find_book_index(title)
    sims = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).ravel()
    sims[idx] = -1                                  # саму книжку не рекомендуємо
    top = np.argsort(sims)[::-1][:n]
    out = books.loc[top, ["book_id", "title", "authors"]].copy()
    out["similarity"] = sims[top].round(4)
    return out.reset_index(drop=True)

recommend("The Hunger Games")

,book_id,title,authors,similarity
0,507,The Hunger Games Trilogy Boxset (The Hunger Ga...,Suzanne Collins,0.8471
1,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.7829
2,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.7317
3,6224,The World of the Hunger Games (Hunger Games Tr...,Kate Egan,0.6281
4,1355,The Hunger Games Tribute Guide,Emily Seife,0.4610


In [7]:
example_titles = ["The Hunger Games", "Harry Potter and the Sorcerer's Stone", "The Hobbit"]
frames = []
for t in example_titles:
    r = recommend(t, n=5)
    r.insert(0, "query_title", books.loc[find_book_index(t), "title"])
    frames.append(r)
content_recs = pd.concat(frames, ignore_index=True)
content_recs.to_csv("content_recommendations.csv", index=False, encoding="utf-8-sig")
content_recs

,query_title,book_id,title,authors,similarity
0,"The Hunger Games (The Hunger Games, #1)",507,The Hunger Games Trilogy Boxset (The Hunger Ga...,Suzanne Collins,0.8471
1,"The Hunger Games (The Hunger Games, #1)",20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.7829
2,"The Hunger Games (The Hunger Games, #1)",17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.7317
3,"The Hunger Games (The Hunger Games, #1)",6224,The World of the Hunger Games (Hunger Games Tr...,Kate Egan,0.6281
4,"The Hunger Games (The Hunger Games, #1)",1355,The Hunger Games Tribute Guide,Emily Seife,0.4610
5,Harry Potter and the Sorcerer's Stone (Harry P...,25,Harry Potter and the Deathly Hallows (Harry Po...,"J.K. Rowling, Mary GrandPré",0.6986
6,Harry Potter and the Sorcerer's Stone (Harry P...,23,Harry Potter and the Chamber of Secrets (Harry...,"J.K. Rowling, Mary GrandPré",0.6965
7,Harry Potter and the Sorcerer's Stone (Harry P...,21,Harry Potter and the Order of the Phoenix (Har...,"J.K. Rowling, Mary GrandPré",0.6939
8,Harry Potter and the Sorcerer's Stone (Harry P...,24,Harry Potter and the Goblet of Fire (Harry Pot...,"J.K. Rowling, Mary GrandPré",0.6937
9,Harry Potter and the Sorcerer's Stone (Harry P...,27,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré",0.6909


## Завдання 3. Колаборативна фільтрація (Surprise, SVD)

In [9]:
SAMPLE_USERS = None

cf_df = df[["user_id", "book_id", "rating"]]
if SAMPLE_USERS:
    users = cf_df["user_id"].drop_duplicates().sample(SAMPLE_USERS, random_state=42)
    cf_df = cf_df[cf_df["user_id"].isin(users)]
print("Оцінок для CF:", len(cf_df))

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(cf_df, reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

Оцінок для CF: 5976479


In [10]:
svd = SVD(random_state=42)
svd.fit(trainset)
predictions = svd.test(testset)
rmse = accuracy.rmse(predictions)

RMSE: 0.8310


In [11]:
with open("svd_model.pkl", "wb") as f:
    pickle.dump(svd, f)
print("Модель збережено у svd_model.pkl")

Модель збережено у svd_model.pkl
